In [ ]:
from langgraph.graph import StateGraph, START, END
from pydantic import BaseModel, Field
from typing import Annotated
import operator
from langchain_groq import ChatGroq
import os


In [ ]:
class Worker(BaseModel):
    name:str
    dependency:list[str] = Field(default_factory=list)
    status:str='PENDING'



In [ ]:
class Nodestate(BaseModel):
    completed_tasks: Annotated[
                list[str],
                operator.add
            ] = Field(
                default_factory=list
           )
    result:list[str]=[]
    workers:Annotated[list[Worker],operator.add]=Field(default_factory=list)

In [ ]:
def research_worker(state:Nodestate):
    return{
        state.name:"Research",
        state.dependency:[],
        state.status:"READY"
    }
def finance_review(state:Nodestate):
    return{
        state.name:"FinanceReview",
        state.dependency:[],
        state.status:"READY"
    }
def final_review(state:Nodestate):
    return{
        state.name:"FinalReview",
        state.dependency:['"Research"',"FinanceReview"],
        state.status:'PENDING'
    }
def writer(state:Nodestate):
    prompt="you are an expert review person review my above node"
    result=llm.invoke(prompt)
    return{
        state.dependency:['final_review'],
        state.result:result
    }
def schedular(state:Nodestate):
    for task in state.workers:
        if task.name in task.completed_tasks:
            continue
        dependancy_satisfied=all(
            dep in task.completed_tasks
            for dep in task.dependency
        )
        if dependancy_satisfied:
            task.status='RUNING'
        else:
            task.status='BLOCKED'
    return {
        "status":state.status
    }
def runtime(state:Nodestate):
    for task in state.workers:
        if task.status!="RUNNING":
            continue
        print(
                    f"Executing: {task.name}"
                )
        task.status="SUCCESS"
    completed_now.append(task.name)
    return{
        "Worker":state.name,
        "completed_tasks":completed_now
        }
def remaining_task(state:Nodestate):
    for task in state.workers:
        if task.name not in state.completed_tasks:
            return "Scheduler"
    return "END"

In [ ]:
graph=StateGraph(Nodestate)
graph.add_node("Rsearch",research_worker)
graph.add_node("FinanceReview",finance_review)
graph.add_node("Schedular",schedular)
graph.add_node("Runtime",runtime)
graph.add_edge(START,"Research")
graph.add_edge(START,"FinanceReview")
graph.add_edge("Research","Schedular")
graph.add_edge("FinanceReview","Schedular")
graph.add_edge("Schedular","Runtime")
graph.add_conditional_edges(
    "Runtime",
    remaining_task,
    {
                "Schedular": "Schedular",
                "END": END
    }
)
app = graph.compile()
app

In [ ]:
from langgraph.graph import StateGraph, START, END
from pydantic import BaseModel, Field
from typing import Annotated
import operator
# =========================================================
# 1. TASK MODEL
# =========================================================
class Worker(BaseModel):
    name: str
    dependencies: list[str] = Field(default_factory=list)
    status: str = "PENDING"
    retry_limit=0
    max_retry=2
# =========================================================
# 2. GLOBAL STATE
# =========================================================

class NodeState(BaseModel):

    workers: list[Worker] = Field(
        default_factory=list
    )

    completed_tasks: Annotated[
        list[str],
        operator.add
    ] = Field(
        default_factory=list
    )
    fallback: str ='HITL'


# =========================================================
# 3. CREATE TASKS
# =========================================================

def create_tasks(state: NodeState):

    workers = [

        # No dependency
        Worker(
            name="Research",
            dependencies=[]
        ),

        # No dependency
        Worker(
            name="FinanceReview",
            dependencies=[],
            status="FAILED"
        ),

        # Depends on BOTH Research and FinanceReview
        Worker(
            name="FinalReview",
            dependencies=[
                "Research",
                "FinanceReview"
            ]
        ),

        # Depends on FinalReview
        Worker(
            name="Writer",
            dependencies=[
                "FinalReview"
            ]
        )
    ]

    print("\n========== TASKS CREATED ==========\n")

    for task in workers:
        print(
            f"{task.name:15} | "
            f"Dependencies: {task.dependencies} | "
            f"Status: {task.status}"
        )

    return {
        "workers": workers
    }


# =========================================================
# 4. SCHEDULER
# =========================================================

def scheduler(state: NodeState):

    print("\n========== SCHEDULER ==========\n")

    ready_tasks = []

    for task in state.workers:

        # Already completed task ko skip karo
        if task.name in state.completed_tasks:
            continue
        # ---------------------------------------------
        # DEPENDENCY CHECK
        # ---------------------------------------------
        dependencies_satisfied = all(
            dependency in state.completed_tasks
            for dependency in task.dependencies
        )
        # ---------------------------------------------
        # READY
        # ---------------------------------------------
        if dependencies_satisfied:
            task.status = "READY"
            ready_tasks.append(task)
        # ---------------------------------------------
        # BLOCKED
        # ---------------------------------------------
        else:

            task.status = "BLOCKED"

    # =====================================================
    # CONCURRENCY LIMIT
    # =====================================================

    concurrency_limit = 2

    running_tasks = ready_tasks[:concurrency_limit]

    waiting_tasks = ready_tasks[concurrency_limit:]

    # =====================================================
    # RUNNING
    # =====================================================

    for task in running_tasks:
        task.status = "RUNNING"

    # =====================================================
    # WAITING
    # =====================================================

    for task in waiting_tasks:
        task.status = "WAITING"

    # =====================================================
    # PRINT
    # =====================================================

    for task in state.workers:

        print(
            f"{task.name:15} | "
            f"Dependencies: {task.dependencies} | "
            f"Status: {task.status}"
        )

    return {
        "workers": state.workers
    }

def retry(state:Nodestate):
  
    for task in state.workers:
        if retry_limit<task.max_retry:
            task.status="READY"
            retry_limit +=1
        else:
            return "HITL"
def hitl(state:Nodestate):
    user_input=input("Please give me reponse either its True or False ")
    return{
        "fallback":user_input
    }
        
# =========================================================
# 5. RUNTIME
# =========================================================

def runtime(state: NodeState):

    print("\n========== RUNTIME ==========\n")

    completed_now = []
    failed_now=[]

    for task in state.workers:

        # Sirf RUNNING tasks execute honge
        if task.status != "RUNNING":
            continue

        print(
            f"Executing: {task.name}"
        )

        # Simulated successful execution
        task.status = "SUCCESS"

        completed_now.append(task.name)

        print(
            f"{task.name} -> SUCCESS"
        )
        if task.status=="FAILED":
            failed_now.append(task.name)
            return "RETRY"

    return {
        "workers": state.workers,
        "completed_tasks": completed_now
    }


# =========================================================
# 6. CHECK REMAINING TASKS
# =========================================================

def check_remaining_tasks(state: NodeState):

    for task in state.workers:

        if task.name not in state.completed_tasks:

            return "Scheduler"

    return "END"


# =========================================================
# 7. BUILD GRAPH
# =========================================================

graph = StateGraph(NodeState)

graph.add_node(
    "CreateTasks",
    create_tasks
)

graph.add_node(
    "Scheduler",
    scheduler
)

graph.add_node(
    "Runtime",
    runtime
)
graph.add_node("RETRY",retry)
graph.add_node("HITL",hitl)
# =========================================================
# 8. GRAPH FLOW
# =========================================================

graph.add_edge(
    START,
    "CreateTasks"
)

graph.add_edge(
    "CreateTasks",
    "Scheduler"
)

graph.add_edge(
    "Scheduler",
    "Runtime"
)


# =========================================================
# 9. CONDITIONAL ROUTING
# =========================================================

graph.add_conditional_edges(
    "Runtime",
    check_remaining_tasks,
    {
        "Scheduler": "Scheduler",
        "END": END
    }
)
graph.add_conditional_edges(
    "Runtime",
    retry,
    {
        "RETRY":"RETRY",
    }
)
graph.add_conditional_edges(
    "Runtime",
    hitl,
    {
        "HITL":"HITL"
    }
)


# =========================================================
# 10. COMPILE
# =========================================================

app = graph.compile()


# =========================================================
# 11. EXECUTE
# =========================================================

result = app.invoke({})


# =========================================================
# 12. FINAL STATE
# =========================================================

print("\n========== FINAL STATE ==========\n")

print(
    "Completed Tasks:",
    result["completed_tasks"]
)

print()

for task in result["workers"]:

    print(
        f"Task: {task.name}"
    )

    print(
        f"Dependencies: {task.dependencies}"
    )

    print(
        f"Status: {task.status}"
    )
    print("-----------------------------------")